# LAB Multi-Phase Training Tutorial

This notebook walks through **LAB (Large-scale Alignment for chatBots) multi-phase training** using the Training Hub library. The pipeline has two phases:

1. **Phase07 — Knowledge Tuning**: Train on knowledge-heavy data to build foundational understanding
2. **Phase10 — Skills + Replay**: Train on skills data with replay of Phase07 knowledge data **and** the base model's original instruction-tuning data

This approach first establishes knowledge foundations, then adds task-specific skills while preventing forgetting through comprehensive replay.

> For long-running jobs, consider the script version at `lab_multiphase_training.py` for better logging and resumability.

## Setup and Imports

In [ ]:
from training_hub import sft

import os
import time
import logging
from datetime import datetime

## Logging Configuration

Reduce noise from library loggers while preserving training progress output.

In [ ]:
for name in ("transformers", "torch", "accelerate"):
    logging.getLogger(name).setLevel(logging.WARNING)


def run_training(training_func, description="Training"):
    """Run a training function with timing and error handling."""
    env_overrides = {
        "TRANSFORMERS_VERBOSITY": "warning",
        "TOKENIZERS_PARALLELISM": "false",
    }
    saved = {k: os.environ.get(k) for k in env_overrides}
    os.environ.update(env_overrides)

    print(f"Starting {description} ...")
    t0 = time.time()
    try:
        result = training_func()
        elapsed = time.time() - t0
        print(f"{description} completed in {elapsed / 3600:.2f} hours")
        return result
    except Exception as exc:
        elapsed = time.time() - t0
        print(f"{description} failed after {elapsed / 60:.1f} minutes: {exc}")
        raise
    finally:
        for k, v in saved.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v


def find_latest_checkpoint(ckpt_output_dir):
    """Return the path to the most recently created checkpoint."""
    hf_dir = os.path.join(ckpt_output_dir, "hf_format")
    if not os.path.isdir(hf_dir):
        return None
    entries = [
        os.path.join(hf_dir, d)
        for d in os.listdir(hf_dir)
        if os.path.isdir(os.path.join(hf_dir, d))
    ]
    return max(entries, key=os.path.getctime) if entries else None

## Configuration

Update the paths below to match your environment.

In [ ]:
# ---- Paths ----
base_model_path   = "/path/to/your/base/model"           # e.g., granite-3.1-8b-starter-v2.1
phase07_data_path = "/path/to/knowledge_data.jsonl"       # Knowledge data for Phase07
phase10_data_path = "/path/to/skills_plus_replay.jsonl"   # Skills + replay data for Phase10
ckpt_output_base  = "/path/to/your/checkpoints"

# ---- Training hyperparameters ----
max_tokens_per_gpu = 25_000
max_seq_len        = 20_000

# ---- Distributed training ----
nproc_per_node = 8
nnodes         = 1
node_rank      = 0
rdzv_id        = 47
rdzv_endpoint  = "0.0.0.0:12345"

print(f"Base model:        {base_model_path}")
print(f"Phase07 data:      {phase07_data_path}")
print(f"Phase10 data:      {phase10_data_path}")
print(f"GPUs per node:     {nproc_per_node}")
print(f"Tokens per GPU:    {max_tokens_per_gpu:,}")
print(f"\nPhase07 uses batch_size=128  (smaller, focused knowledge dataset)")
print(f"Phase10 uses batch_size=3840 (larger, combined skills + replay dataset)")

## Phase 1: Knowledge Tuning (Phase07)

Train the base model on knowledge-heavy data — facts, domain knowledge, core concepts — to establish a strong knowledge foundation before adding skills.

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
phase07_ckpt_dir = os.path.join(ckpt_output_base, f"lab_phase07_{timestamp}")

print(f"Phase07 output: {phase07_ckpt_dir}")


def phase07_training():
    return sft(
        model_path=base_model_path,
        data_path=phase07_data_path,
        ckpt_output_dir=phase07_ckpt_dir,
        num_epochs=7,
        effective_batch_size=128,
        learning_rate=2e-5,
        max_seq_len=max_seq_len,
        max_tokens_per_gpu=max_tokens_per_gpu,
        data_output_dir="/dev/shm",
        warmup_steps=0,
        save_samples=0,
        checkpoint_at_epoch=True,
        accelerate_full_state_at_epoch=False,
        nproc_per_node=nproc_per_node,
        nnodes=nnodes,
        node_rank=node_rank,
        rdzv_id=rdzv_id,
        rdzv_endpoint=rdzv_endpoint,
    )


run_training(phase07_training, "Phase07 (Knowledge Tuning)")

## Checkpoint Discovery

Find the latest Phase07 checkpoint to use as the starting model for Phase10.

In [ ]:
phase07_checkpoint = find_latest_checkpoint(phase07_ckpt_dir)

if phase07_checkpoint:
    print(f"Latest Phase07 checkpoint: {phase07_checkpoint}")
else:
    raise RuntimeError(
        f"No Phase07 checkpoint found in {phase07_ckpt_dir}/hf_format. "
        "Ensure Phase07 completed successfully before continuing."
    )

## Phase 2: Skills + Replay (Phase10)

Continue from the Phase07 checkpoint using a combined dataset that includes:

- **New skills data** — task instructions, problem-solving examples
- **Phase07 knowledge replay** — prevents forgetting newly acquired knowledge
- **Base model instruction replay** — preserves original instruction-following capabilities

In [ ]:
phase10_ckpt_dir = os.path.join(ckpt_output_base, f"lab_phase10_{timestamp}")

print(f"Phase10 input model: {phase07_checkpoint}")
print(f"Phase10 output:      {phase10_ckpt_dir}")


def phase10_training():
    return sft(
        model_path=phase07_checkpoint,
        data_path=phase10_data_path,
        ckpt_output_dir=phase10_ckpt_dir,
        num_epochs=7,
        effective_batch_size=3840,
        learning_rate=2e-5,
        max_seq_len=max_seq_len,
        max_tokens_per_gpu=max_tokens_per_gpu,
        data_output_dir="/dev/shm",
        warmup_steps=0,
        save_samples=0,
        checkpoint_at_epoch=True,
        accelerate_full_state_at_epoch=True,
        nproc_per_node=nproc_per_node,
        nnodes=nnodes,
        node_rank=node_rank,
        rdzv_id=rdzv_id,
        rdzv_endpoint=rdzv_endpoint,
    )


run_training(phase10_training, "Phase10 (Skills + Replay)")

## Training Summary

In [ ]:
print("LAB Multi-Phase Training Summary")
print("=" * 50)
print(f"Phase07 output: {phase07_ckpt_dir}")
print(f"Phase10 output: {phase10_ckpt_dir}")

final_ckpt = find_latest_checkpoint(phase10_ckpt_dir)
if final_ckpt:
    print(f"\nFinal model: {final_ckpt}")

print(f"\nConfiguration:")
print(f"  Max tokens/GPU:    {max_tokens_per_gpu:,}")
print(f"  Max seq len:       {max_seq_len:,}")
print(f"  GPUs/node:         {nproc_per_node}")
print(f"  Phase07 batch:     128")
print(f"  Phase10 batch:     3840")
print(f"  Learning rate:     2e-5")
print(f"  Epochs/phase:      7")

print(f"\nNext steps:")
print(f"  1. Evaluate on relevant benchmarks")
print(f"  2. Test knowledge retention from Phase07")
print(f"  3. Verify skills acquisition from Phase10")
print(f"  4. Confirm base instruction-following is preserved")
print(f"  5. Deploy for inference")

## Key Concepts

### Why Two Phases?

| Phase | Purpose | Batch Size | Dataset |
|-------|---------|------------|--------|
| Phase07 | Knowledge acquisition | 128 (small) | Focused knowledge data |
| Phase10 | Skills + preservation | 3840 (large) | Skills + knowledge replay + instruction replay |

### Phase10 Replay Strategy

The Phase10 dataset is a carefully balanced mixture of three components:

1. **New skills/task data** — primary learning objective
2. **Phase07 knowledge replay** — prevents knowledge drift
3. **Base model instruction replay** — preserves original capabilities

### Troubleshooting

| Issue | Solution |
|-------|----------|
| OOM errors | Reduce `max_tokens_per_gpu` or `effective_batch_size` |
| Checkpoint not found | Verify Phase07 completed; check `ckpt_output_dir` permissions |
| Distributed errors | Verify network connectivity; check `rdzv_endpoint` |
| Data loading errors | Verify JSONL paths exist and format is valid |